# iProteinHunter Pipeline Control Notebook

Use this notebook to configure and launch `iproteinhunter_run.sh` without typing long CLI commands manually.

This notebook supports:
- selecting predictor and post-predictor tools
- setting run counts, cycle counts, binder length range
- auto/manual parallel controls
- launching with `caffeinate -dims`
- lightweight result summaries from output CSVs
Kernel: select **iProteinHunter Notebook** in VS Code before running cells.



## 1) Setup

Run the next cell first. It auto-detects the repository root and validates the runner path.


In [ ]:
from __future__ import annotations

from pathlib import Path
import csv
import datetime as dt
import shlex
import statistics
import subprocess
from typing import Any

# Resolve repo root whether notebook is run from repo root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / "iproteinhunter_run.sh").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "iproteinhunter_run.sh").exists():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate iproteinhunter_run.sh from current working directory.")

RUNNER = REPO_ROOT / "iproteinhunter_run.sh"
EXAMPLES_DIR = REPO_ROOT / "examples"
OUTPUT_ROOT = REPO_ROOT / "output"

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"RUNNER:    {RUNNER}")
print(f"EXAMPLES:  {EXAMPLES_DIR}")
print(f"OUTPUT:    {OUTPUT_ROOT}")


## 2) Configure your run

Edit values in `cfg` and run this cell.

Notes:
- `predictor`: `boltz`, `intellifold`, or `openfold-3-mlx`
- `post_predictor`: comma-separated list like `boltz,openfold-3-mlx`
- `max_parallel`: integer or `"auto"`
- if `mps_aware=False`, `--no-mps-aware` is passed


In [ ]:
cfg: dict[str, Any] = {
    # Core
    "predictor": "boltz",
    "post_predictor": "intellifold,openfold-3-mlx",
    "post_mode": "all",
    "template_yaml": str(EXAMPLES_DIR / "aCbx_bind.yaml"),
    "run_name": f"notebook_run_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}",
    "out_root": str(OUTPUT_ROOT),

    # Scale
    "num_runs": 10,
    "num_opt_cycles": 5,
    "binder_min_len": 65,
    "binder_max_len": 150,

    # Parallel controls
    "max_parallel": "auto",    # int or "auto"
    "mps_aware": True,

    # Launch behavior
    "use_caffeinate": True,
    "background": True,
}

cfg


## 3) Build command and launch

- First call prints the exact command.
- Second call launches in background by default and writes a launcher log file.


In [ ]:
def build_cmd(cfg: dict[str, Any]) -> list[str]:
    cmd = [
        str(RUNNER),
        "--predictor", str(cfg["predictor"]),
        "--post-predictor", str(cfg["post_predictor"]),
        "--post-mode", str(cfg.get("post_mode", "all")),
        "--run-name", str(cfg["run_name"]),
        "--template-yaml", str(cfg["template_yaml"]),
        "--out-root", str(cfg.get("out_root", OUTPUT_ROOT)),
        "--num-runs", str(cfg["num_runs"]),
        "--num-opt-cycles", str(cfg["num_opt_cycles"]),
        "--binder-min-len", str(cfg["binder_min_len"]),
        "--binder-max-len", str(cfg["binder_max_len"]),
        "--max-parallel", str(cfg.get("max_parallel", "auto")),
    ]

    if not bool(cfg.get("mps_aware", True)):
        cmd.append("--no-mps-aware")

    return cmd


def launch(cfg: dict[str, Any]) -> dict[str, Any]:
    cmd = build_cmd(cfg)
    if bool(cfg.get("use_caffeinate", True)):
        cmd = ["caffeinate", "-dims", *cmd]

    out_root = Path(cfg.get("out_root", OUTPUT_ROOT)).resolve()
    run_name = str(cfg["run_name"])
    launch_dir = out_root / run_name
    launch_dir.mkdir(parents=True, exist_ok=True)
    launcher_log = launch_dir / "launcher.log"

    print("Command:")
    print("  " + shlex.join(cmd))
    print(f"Launcher log: {launcher_log}")

    if bool(cfg.get("background", True)):
        fh = open(launcher_log, "a", buffering=1)
        proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, start_new_session=True)
        return {
            "pid": proc.pid,
            "launcher_log": str(launcher_log),
            "run_root": str(Path(cfg.get("out_root", OUTPUT_ROOT)) / run_name),
            "command": cmd,
            "background": True,
        }

    # foreground mode
    subprocess.run(cmd, check=True)
    return {
        "pid": None,
        "launcher_log": str(launcher_log),
        "run_root": str(Path(cfg.get("out_root", OUTPUT_ROOT)) / run_name),
        "command": cmd,
        "background": False,
    }


# Launch (assign result to keep run metadata handy)
launch_info = launch(cfg)
launch_info


## 4) Monitor and summarize

Use these helper functions after launch.


In [ ]:
def tail_log(path: str | Path, n: int = 40) -> None:
    p = Path(path)
    if not p.exists():
        print(f"Log not found: {p}")
        return
    lines = p.read_text(errors="ignore").splitlines()
    for line in lines[-n:]:
        print(line)


def process_alive(pid: int | None) -> bool:
    if pid is None:
        return False
    rc = subprocess.run(["ps", "-p", str(pid)], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return rc.returncode == 0


def summarize_scores(run_root: str | Path) -> None:
    rr = Path(run_root)
    summary = rr / "summary_all_runs.csv"
    if not summary.exists():
        print(f"Summary not ready: {summary}")
        return

    with summary.open(newline="") as f:
        rows = list(csv.DictReader(f))

    iptms = []
    for r in rows:
        try:
            iptms.append(float(r["iptm"]))
        except Exception:
            pass

    print(f"Rows: {len(rows)}")
    if iptms:
        print(f"iPTM mean:   {statistics.fmean(iptms):.4f}")
        print(f"iPTM median: {statistics.median(iptms):.4f}")
        print(f"iPTM max:    {max(iptms):.4f}")


# Example usage after launch:
if "launch_info" in globals():
    print("Alive:", process_alive(launch_info.get("pid")))
    tail_log(launch_info.get("launcher_log"), n=25)
    summarize_scores(launch_info.get("run_root"))


## 5) Useful presets

Quickly switch configs for common workflows.


In [ ]:
def preset_acbx_10_auto() -> dict[str, Any]:
    return {
        "predictor": "boltz",
        "post_predictor": "intellifold,openfold-3-mlx",
        "post_mode": "all",
        "template_yaml": str(EXAMPLES_DIR / "aCbx_bind.yaml"),
        "run_name": f"acbx_10_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}",
        "out_root": str(OUTPUT_ROOT),
        "num_runs": 10,
        "num_opt_cycles": 5,
        "binder_min_len": 65,
        "binder_max_len": 150,
        "max_parallel": "auto",
        "mps_aware": True,
        "use_caffeinate": True,
        "background": True,
    }


def preset_intellifold_100() -> dict[str, Any]:
    d = preset_acbx_10_auto()
    d["predictor"] = "intellifold"
    d["post_predictor"] = "boltz,openfold-3-mlx"
    d["num_runs"] = 100
    d["run_name"] = f"intellifold_100_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    return d


# Example:
# cfg = preset_acbx_10_auto()
# cfg = preset_intellifold_100()
# cfg


## 6) Interactive structure browser (ipywidgets + py2Dmol)

This section scans `<run_root>/cifs_all`, matches each CIF with `summary_all_runs.csv`, and lets you inspect structures interactively.

If `py2Dmol` is unavailable, it falls back to `py3Dmol` (if installed).


In [ ]:
# Optional one-time install (uncomment if needed in your notebook kernel)
# %pip install ipywidgets py2Dmol py3Dmol

import re
from pathlib import Path
import csv

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output


In [ ]:
def resolve_run_root_for_viewer() -> Path:
    if 'launch_info' in globals() and isinstance(launch_info, dict):
        rr = launch_info.get('run_root')
        if rr:
            p = Path(rr)
            if p.exists():
                return p

    if 'cfg' in globals() and isinstance(cfg, dict):
        rr = Path(cfg.get('out_root', OUTPUT_ROOT)) / str(cfg.get('run_name', ''))
        if rr.exists():
            return rr

    out = Path(OUTPUT_ROOT)
    if out.exists():
        dirs = sorted([d for d in out.iterdir() if d.is_dir()], key=lambda d: d.stat().st_mtime, reverse=True)
        if dirs:
            return dirs[0]

    raise FileNotFoundError('Could not resolve run root. Set run_root_text manually.')


def parse_summary(summary_csv: Path):
    by_key = {}
    if not summary_csv.exists():
        return by_key

    with summary_csv.open(newline='') as f:
        for row in csv.DictReader(f):
            try:
                run_i = int(str(row.get('run', '')).strip())
                cyc_i = int(str(row.get('cycle', '')).strip())
            except Exception:
                continue
            by_key[(run_i, cyc_i)] = {
                'iptm': row.get('iptm', ''),
                'complex_plddt': row.get('complex_plddt', ''),
                'binder_sequence': row.get('binder_sequence', ''),
                'structure_path': row.get('structure_path', ''),
                'confidence_json': row.get('confidence_json', ''),
            }
    return by_key


def collect_cif_entries(run_root: Path):
    cifs_dir = run_root / 'cifs_all'
    summary_csv = run_root / 'summary_all_runs.csv'
    by_key = parse_summary(summary_csv)

    entries = []
    if not cifs_dir.exists():
        return entries

    pat = re.compile(r'run_(\d+)_cycle_(\d+)_model_\d+\.cif$')
    for cif in sorted(cifs_dir.glob('*.cif')):
        m = pat.search(cif.name)
        run_i = None
        cyc_i = None
        if m:
            run_i = int(m.group(1))
            cyc_i = int(m.group(2))
        metrics = by_key.get((run_i, cyc_i), {})
        iptm = metrics.get('iptm', '')
        plddt = metrics.get('complex_plddt', '')

        if run_i is None or cyc_i is None:
            label = f"{cif.name} | iPTM {iptm or 'NA'} | pLDDT {plddt or 'NA'}"
        else:
            label = f"run {run_i:03d} | cycle {cyc_i:02d} | iPTM {iptm or 'NA'} | pLDDT {plddt or 'NA'}"

        entries.append({
            'label': label,
            'path': cif,
            'run': run_i,
            'cycle': cyc_i,
            'metrics': metrics,
        })

    return entries


def make_mol_view(cif_text: str, width: int = 900, height: int = 600):
    errors = []

    try:
        import py2Dmol as p2
        if hasattr(p2, 'view'):
            v = p2.view(width=width, height=height)
            v.addModel(cif_text, 'mmcif')
            v.setStyle({'cartoon': {'color': 'spectrum'}})
            v.zoomTo()
            return v, 'py2Dmol'
        errors.append('py2Dmol imported but no view()')
    except Exception as e:
        errors.append(f'py2Dmol error: {e}')

    try:
        import py3Dmol as p3
        v = p3.view(width=width, height=height)
        v.addModel(cif_text, 'mmcif')
        v.setStyle({'cartoon': {'color': 'spectrum'}})
        v.zoomTo()
        return v, 'py3Dmol'
    except Exception as e:
        errors.append(f'py3Dmol error: {e}')

    return None, ' | '.join(errors)


In [ ]:
try:
    default_root = resolve_run_root_for_viewer()
except Exception:
    default_root = Path(OUTPUT_ROOT)

run_root_text = widgets.Text(
    value=str(default_root),
    description='Run root:',
    layout=widgets.Layout(width='95%')
)
refresh_btn = widgets.Button(description='Refresh list', button_style='info')
entry_dd = widgets.Dropdown(options=[], description='Structure:', layout=widgets.Layout(width='95%'))
show_btn = widgets.Button(description='Show structure', button_style='success')

meta_out = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='8px'))
view_out = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='8px'))

entries_cache = []


def refresh_entries(_=None):
    global entries_cache
    rr = Path(run_root_text.value).expanduser().resolve()
    entries_cache = collect_cif_entries(rr)
    if not entries_cache:
        entry_dd.options = [('No CIF entries found', None)]
        with meta_out:
            clear_output(wait=True)
            display(Markdown(f'**No entries found under:** `{rr}`'))
        with view_out:
            clear_output(wait=True)
        return

    entry_dd.options = [(e['label'], i) for i, e in enumerate(entries_cache)]
    entry_dd.value = 0
    show_entry(None)


def show_entry(_=None):
    idx = entry_dd.value
    if idx is None:
        return
    e = entries_cache[idx]
    cif_path = e['path']
    m = e['metrics']

    with meta_out:
        clear_output(wait=True)
        meta_lines = [
            '### Structure metadata',
            f"- **Run:** `{e['run']}`",
            f"- **Cycle:** `{e['cycle']}`",
            f"- **iPTM:** `{m.get('iptm', 'NA')}`",
            f"- **Complex pLDDT:** `{m.get('complex_plddt', 'NA')}`",
            f"- **CIF:** `{cif_path}`",
            f"- **Confidence JSON:** `{m.get('confidence_json', 'NA')}`",
            f"- **Binder sequence length:** `{len(m.get('binder_sequence', '')) if m.get('binder_sequence') else 'NA'}`",
        ]
        display(Markdown('
'.join(meta_lines)))

    with view_out:
        clear_output(wait=True)
        cif_text = cif_path.read_text(errors='ignore')
        viewer, backend = make_mol_view(cif_text)
        if viewer is None:
            display(Markdown('**Viewer backend unavailable.** Install one of: `py2Dmol`, `py3Dmol`.'))
            display(Markdown(f'`{backend}`'))
            return

        display(Markdown(f'**Viewer backend:** `{backend}`'))
        if hasattr(viewer, 'show'):
            display(viewer.show())
        else:
            display(viewer)


refresh_btn.on_click(refresh_entries)
show_btn.on_click(show_entry)
entry_dd.observe(show_entry, names='value')

ui = widgets.VBox([
    run_root_text,
    widgets.HBox([refresh_btn, show_btn]),
    entry_dd,
    meta_out,
    view_out,
])

display(ui)
refresh_entries()


### Tips

- Point **Run root** to a folder like:
  - `output/<campaign_or_run_name>/<design_run_name>`
- The browser reads:
  - `summary_all_runs.csv`
  - `cifs_all/*.cif`
- Labels are generated as:
  - `run ### | cycle ## | iPTM ... | pLDDT ...`
